# Linear Regression — Statistics

**Goal.** Treat the OLS estimator θ̂ as a *random* vector that depends on the noisy targets y, and quantify its statistical behaviour: bias, variance, optimality (Gauss–Markov), sampling distribution under normal noise, confidence intervals, hypothesis tests, the R² decomposition, and what happens when assumptions break.

**Role of this notebook.** Estimator theory: math first, with **minimal** Monte Carlo simulations used only to *verify* a theorem numerically. The deterministic linear-algebra picture (normal equations, hat matrix, pseudoinverse) is in `02_mathematics.ipynb`; the optimisation algorithms in `03_optimization.ipynb`; the full implementation in `05_hands_on_programming.ipynb`.

**Prerequisites.** `02_mathematics.ipynb` — in particular the closed form θ\* = (XᵀX)⁻¹ Xᵀ y (Theorem 4.2), the hat matrix H (Theorem 5.3), and Pythagoras ‖y‖² = ‖ŷ\*‖² + ‖r\*‖² (Corollary 5.4).

**Stage map.** `01_intuition` → `02_mathematics` → `03_optimization` → **`04_statistics`** → `05_hands_on_programming`.

**Eight questions.**

1. What is random and what is fixed? (The statistical setup.)
2. Under what assumptions is θ̂ a meaningful estimator? (Gauss–Markov A1–A4.)
3. Is θ̂ on average correct? (Unbiasedness.)
4. How much does θ̂ vary across resamples? (Variance formula.)
5. Is θ̂ the *best* linear unbiased estimator? (Gauss–Markov theorem.)
6. What is its full distribution? (Adding Gaussianity, A5.)
7. How do we test hypotheses and build confidence intervals?
8. How much variance does the model explain? (R² via Pythagoras.)

---

**Reading conventions.** Same as `02_mathematics.ipynb`: theorems in blockquotes, derivations in code blocks, equations numbered only when referred to later. Code cells run small Monte Carlo experiments that *verify* a theorem — never derive one.

In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import random

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

plt.rcParams["figure.dpi"] = 90

## 1. The statistical setup

In `02_mathematics.ipynb` the targets y were just *numbers*. To do statistics we must say where those numbers come from. The standard **fixed-design** setup of linear regression goes:

- The design matrix X ∈ ℝⁿˣᵖ is **fixed** (not random). We condition on it throughout — all expectations and variances are conditional on X.
- There exists an **unknown true parameter** θ ∈ ℝᵖ. We never see θ; we only see y.
- The targets y ∈ ℝⁿ are **random**, generated by

> y = X θ + ε,    where ε ∈ ℝⁿ is a random noise vector.   (1.1)

The OLS estimator

> θ̂ := (XᵀX)⁻¹ Xᵀ y   (assuming rank(X) = p; see Theorem 4.2 of `02_mathematics.ipynb`)

is a random vector because it is a (deterministic) function of the random vector y. Its randomness is inherited entirely from ε.

**What we want.** A list of *properties* of θ̂ — its mean, variance, distribution — written in terms of the *assumptions* we put on ε. Sections 2–6 do exactly that.

## 2. The Gauss–Markov assumptions

Classical regression theory is organised around five assumptions on (X, ε). The first four are the **Gauss–Markov** assumptions; the fifth (normality) is needed only for exact distributional results.

| Label | Name | Statement |
|---|---|---|
| **A1** | Linearity in parameters | y = X θ + ε with X ∈ ℝⁿˣᵖ fixed, rank(X) = p. |
| **A2** | Strict exogeneity (zero mean noise) | 𝔼[ε] = 0. |
| **A3** | Homoscedasticity | Var(εᵢ) = σ² for every i. |
| **A4** | No autocorrelation | Cov(εᵢ, εⱼ) = 0 for i ≠ j. |
| **A5** | Normality (optional) | ε ~ 𝒩(0, σ² · Iₙ). |

**Compact restatement of A2 + A3 + A4.** All three together are equivalent to a single matrix identity on the covariance of the noise:

> 𝔼[ε] = 0,    Var(ε) := 𝔼[ ε εᵀ ] = σ² · Iₙ.   (2.1)

**Compact restatement of A2 + A3 + A4 + A5.** Adding normality gives the joint distribution

> ε ~ 𝒩(0, σ² · Iₙ).   (2.2)

(2.2) is exactly the assumption of Theorem 2.2 in `02_mathematics.ipynb` that made OLS = MLE.

## 3. Unbiasedness

> **Theorem 3.1 (Unbiasedness).** Under A1 and A2,
>
> 𝔼[ θ̂ ]  =  θ.

**Proof.** Substitute y = X θ + ε from (1.1) into the closed form:

```
θ̂  =  (XᵀX)⁻¹ Xᵀ y
    =  (XᵀX)⁻¹ Xᵀ (X θ + ε)
    =  (XᵀX)⁻¹ (XᵀX) θ  +  (XᵀX)⁻¹ Xᵀ ε
    =  θ  +  (XᵀX)⁻¹ Xᵀ ε.                              (3.1)
```

Equation (3.1) is the **sampling identity**: θ̂ equals the truth plus a *linear function of the noise*. Take expectation, using linearity and A2 (𝔼[ε] = 0):

```
𝔼[ θ̂ ]  =  θ  +  (XᵀX)⁻¹ Xᵀ · 𝔼[ε]  =  θ  +  0  =  θ.   ∎
```

**Reading.** OLS does not systematically over- or under-estimate. Average over infinitely many fresh draws of y (with X held fixed) and you recover θ exactly. This is what *unbiased* means; it says nothing about how close any single θ̂ is to θ — that is variance, §4.

### 3.1 Monte Carlo verification

Pick a true θ, draw M independent y's, fit OLS each time, and check the empirical mean of θ̂ across draws is close to θ.

In [2]:
n, p = 200, 3
theta_true = np.array([1.5, -2.0, 0.7])
sigma = 0.5

# Fixed design (drawn once and held throughout).
X = np.column_stack([np.ones(n), rng.normal(size=n), rng.normal(size=n)])

M = 5000  # number of Monte Carlo replications
thetas = np.empty((M, p))
for m in range(M):
    eps = rng.normal(0, sigma, size=n)
    y = X @ theta_true + eps
    thetas[m] = np.linalg.solve(X.T @ X, X.T @ y)

emp_mean = thetas.mean(axis=0)
print(f"true theta            = {theta_true}")
print(f"empirical mean theta_hat = {emp_mean}")
print(f"max abs deviation     = {np.max(np.abs(emp_mean - theta_true)):.4f}")

true theta            = [ 1.5 -2.   0.7]
empirical mean theta_hat = [ 1.50005418 -1.99934342  0.70035504]
max abs deviation     = 0.0007


**Reading.** With M = 5000 draws the deviation is on the order of 10⁻², consistent with the O(1/√M) Monte Carlo error. The closer the empirical mean to θ_true, the more confidently we say "θ̂ is unbiased".

## 4. Variance of θ̂

> **Theorem 4.1 (Sandwich identity).** Under A1 + A2 + A3 + A4,
>
> Var(θ̂)  =  σ² · (XᵀX)⁻¹.   (4.1)

**Proof.** From the sampling identity (3.1), θ̂ − θ = A ε where A := (XᵀX)⁻¹ Xᵀ. Standard rule for the variance of a linear function of a random vector:

```
Var(θ̂)  =  Var(A ε)  =  A · Var(ε) · Aᵀ.
```

By (2.1), Var(ε) = σ² Iₙ. Therefore

```
Var(θ̂)  =  A · (σ² Iₙ) · Aᵀ
         =  σ² · (XᵀX)⁻¹ Xᵀ · X (XᵀX)⁻¹
         =  σ² · (XᵀX)⁻¹ · (XᵀX) · (XᵀX)⁻¹
         =  σ² · (XᵀX)⁻¹.    ∎
```

**Reading.** Three knobs control the variance of the j-th coefficient — Var(θ̂ⱼ) = σ² · [(XᵀX)⁻¹]ⱼⱼ:

- **σ².** More noise in y ⇒ more noise in θ̂. Linear.
- **Sample size n.** Hidden in XᵀX. If columns of X have bounded entries, (XᵀX) ∝ n, so (XᵀX)⁻¹ ∝ 1/n and the standard error of each coefficient shrinks like 1/√n.
- **Multicollinearity.** When two columns of X are nearly proportional, XᵀX is ill-conditioned and (XᵀX)⁻¹ has huge diagonal entries — the **variance inflation** phenomenon.

### 4.1 Monte Carlo verification

Compute the empirical covariance of θ̂ across the M draws from §3.1 and compare to the theoretical σ² · (XᵀX)⁻¹.

In [3]:
emp_cov = np.cov(thetas, rowvar=False)
theo_cov = sigma**2 * np.linalg.inv(X.T @ X)

print("empirical Var(theta_hat):")
print(emp_cov)
print("\ntheoretical sigma^2 (X^T X)^-1:")
print(theo_cov)
print(f"\nmax abs entrywise difference = {np.max(np.abs(emp_cov - theo_cov)):.5f}")

empirical Var(theta_hat):
[[ 1.23255965e-03  4.69936840e-05 -3.72240218e-05]
 [ 4.69936840e-05  1.54043720e-03  8.87916725e-05]
 [-3.72240218e-05  8.87916725e-05  1.22203280e-03]]

theoretical sigma^2 (X^T X)^-1:
[[ 1.25187414e-03  4.74467096e-05 -2.13847200e-05]
 [ 4.74467096e-05  1.62316674e-03  9.86120027e-05]
 [-2.13847200e-05  9.86120027e-05  1.21466257e-03]]

max abs entrywise difference = 0.00008


## 5. The Gauss–Markov theorem: OLS is BLUE

Unbiasedness alone is not impressive — the trivial estimator "always predict 0" is also a (very bad) function of y. The Gauss–Markov theorem makes a sharper claim: among **all** linear unbiased estimators, OLS has the smallest variance, coordinate by coordinate.

### 5.1 Definitions

> **Definition (linear estimator).** An estimator θ̃ of θ is **linear (in y)** if there exists a matrix C ∈ ℝᵖˣⁿ (depending on X but not on y) such that
>
> θ̃  =  C y.

OLS is linear with C_OLS := (XᵀX)⁻¹ Xᵀ.

> **Definition (unbiased).** θ̃ = C y is **unbiased** if 𝔼[θ̃] = θ for every value of θ ∈ ℝᵖ. Substituting y = X θ + ε and 𝔼[ε] = 0, this is equivalent to the algebraic condition
>
> C X  =  Iₚ.   (5.1)

**Comparison rule.** For two p × p covariance matrices Σ₁, Σ₂, we write Σ₁ ⪯ Σ₂ if Σ₂ − Σ₁ is PSD. Setting v = eⱼ in vᵀ (Σ₂ − Σ₁) v ≥ 0 shows that Σ₁ ⪯ Σ₂ implies the j-th diagonal of Σ₁ is no larger than that of Σ₂ — every individual coordinate has smaller (or equal) variance.

### 5.2 Theorem (Gauss–Markov)

> **Theorem 5.2.** Under A1 + A2 + A3 + A4, the OLS estimator θ̂ is **BLUE** — the *best linear unbiased estimator*. Concretely: for any linear unbiased θ̃ = C y,
>
> Var(θ̂)  ⪯  Var(θ̃),
>
> with equality iff C = C_OLS = (XᵀX)⁻¹ Xᵀ.

**Proof.** Write C = C_OLS + D where D := C − C_OLS. Unbiasedness of both θ̂ and θ̃ requires C X = Iₚ and C_OLS X = Iₚ, hence

```
D X  =  C X − C_OLS X  =  Iₚ − Iₚ  =  0.   (5.2)
```

Compute Var(θ̃) by the same rule as in §4 (Var(C ε) = σ² C Cᵀ since Var(ε) = σ² Iₙ):

```
Var(θ̃)  =  σ² · C Cᵀ
         =  σ² · (C_OLS + D) (C_OLS + D)ᵀ
         =  σ² · [ C_OLS C_OLSᵀ  +  C_OLS Dᵀ  +  D C_OLSᵀ  +  D Dᵀ ].
```

The cross terms vanish because (5.2) gives D X = 0:

```
C_OLS Dᵀ  =  (XᵀX)⁻¹ Xᵀ Dᵀ  =  (XᵀX)⁻¹ (D X)ᵀ  =  0,
D C_OLSᵀ  =  ( C_OLS Dᵀ )ᵀ  =  0.
```

Also C_OLS C_OLSᵀ = (XᵀX)⁻¹ (computed already in §4). Hence

```
Var(θ̃)  =  σ² · (XᵀX)⁻¹  +  σ² · D Dᵀ
         =  Var(θ̂)  +  σ² · D Dᵀ.
```

D Dᵀ is PSD (it is a Gram matrix), so Var(θ̃) − Var(θ̂) ⪰ 0 — exactly the claim. Equality forces D Dᵀ = 0, hence D = 0, hence C = C_OLS. ∎

**Reading.** Gauss–Markov is sharp: OLS is *the* unique BLUE estimator. The catch is the word **linear** — biased or non-linear estimators (e.g., Ridge regression, the Lasso) can have **smaller** variance, at the price of bias. That trade-off is the subject of the bias–variance decomposition (`00_foundations/04_model_evaluation/`) and motivates the next algorithms in module 01.

## 6. Estimating σ² and the sampling distribution

The variance formula (4.1) contains σ², which we do not know. We estimate it from the residuals.

### 6.1 Definition (residual sum of squares and residual variance)

Let r̂ := y − X θ̂ ∈ ℝⁿ be the fitted residual vector. Define

> RSS  :=  ‖r̂‖²  =  ∑ᵢ r̂ᵢ²,
>
> s²   :=  RSS / (n − p).   (6.1)

### 6.2 Theorem (unbiased estimator of σ²)

> **Theorem 6.2.** Under A1 + A2 + A3 + A4, with rank(X) = p,
>
> 𝔼[ s² ]  =  σ²,    so s² is an unbiased estimator of σ².

**Proof sketch.** Using r̂ = (Iₙ − H) y (with H the hat matrix from §5 of `02_mathematics.ipynb`) and the trace identity 𝔼[εᵀ M ε] = σ² · trace(M) for symmetric M and ε with covariance σ² Iₙ:

```
𝔼[ RSS ]  =  𝔼[ εᵀ (Iₙ − H) ε ]  =  σ² · trace(Iₙ − H)  =  σ² · (n − p),
```

since trace(H) = rank(H) = p by Theorem 5.3 of `02_mathematics.ipynb`. Dividing by n − p gives 𝔼[s²] = σ². ∎

**Reading.** The divisor is **n − p**, not n. Each fitted parameter "absorbs" one residual degree of freedom: the n residuals satisfy p linear constraints (Xᵀ r̂ = 0 from the normal equations), so only n − p of them are free. Dividing by n would *under*estimate σ² systematically.

### 6.3 Sampling distribution under normality (adding A5)

Adding A5 (ε ~ 𝒩(0, σ² Iₙ)) makes θ̂ a linear function of a Gaussian, hence Gaussian itself:

> **Theorem 6.3.** Under A1 + A2 + A3 + A4 + A5,
>
> θ̂  ~  𝒩( θ,  σ² · (XᵀX)⁻¹ ),     and     RSS / σ²  ~  χ²(n − p),
>
> with θ̂ and RSS independent.

**Proof sketch.** From (3.1), θ̂ = θ + (XᵀX)⁻¹ Xᵀ ε is an affine function of the Gaussian vector ε, hence Gaussian with the mean and covariance computed in §3 and §4. The residual vector r̂ = (Iₙ − H) ε is also Gaussian; (Iₙ − H) is an orthogonal projection of rank n − p, so ‖r̂‖² / σ² = εᵀ (Iₙ − H) ε / σ² has a χ²(n − p) distribution. Independence of θ̂ and r̂ follows from the orthogonality of their generating projections (H and Iₙ − H), which makes their covariance zero — and for Gaussians, zero covariance implies independence. ∎

**This is the key result for the rest of the notebook.** Sections 7 and 8 are just corollaries: confidence intervals follow from "θ̂ is normal"; the t-test follows from "θ̂ normal + s² independent χ²/(n − p)".

## 7. Confidence intervals

### 7.1 Definition (standard error)

The **standard error** of θ̂ⱼ is the square root of its variance estimate:

> SE(θ̂ⱼ)  :=  s · √[ (XᵀX)⁻¹ ]ⱼⱼ,    j = 1, …, p.   (7.1)

### 7.2 Theorem (t-pivot)

> **Theorem 7.2.** Under A1–A5, for each coordinate j,
>
> ( θ̂ⱼ − θⱼ )  /  SE(θ̂ⱼ)   ~   t(n − p).

**Proof.** Let Vⱼ := [ (XᵀX)⁻¹ ]ⱼⱼ. By Theorem 6.3, (θ̂ⱼ − θⱼ)/(σ √Vⱼ) ~ 𝒩(0, 1) and is independent of (n − p) s²/σ² ~ χ²(n − p). The ratio of a standard normal to √(χ²/df) is the textbook definition of Student's t with df = n − p:

```
( θ̂ⱼ − θⱼ ) / (σ √Vⱼ)                    𝒩(0, 1)
─────────────────────────────  =   ────────────────────────  ~  t(n − p).      ∎
√( s² / σ² )                       √( χ²(n − p) / (n − p) )
```

### 7.3 Construction

Inverting the t-pivot: a (1 − α) confidence interval for θⱼ is

> θ̂ⱼ  ±  t_{1 − α/2}(n − p) · SE(θ̂ⱼ).   (7.2)

**Interpretation.** Across repeated draws of y (with X fixed), the random interval in (7.2) covers the fixed true θⱼ with probability 1 − α.

### 7.4 Monte Carlo coverage check

If the CI is built correctly, 95 % of intervals should cover the true value. We measure this empirically.

In [4]:
alpha = 0.05
t_crit = stats.t.ppf(1 - alpha / 2, df=n - p)

covered = np.zeros(p, dtype=int)
XtX_inv = np.linalg.inv(X.T @ X)
diag_V = np.diag(XtX_inv)

for m in range(M):
    eps = rng.normal(0, sigma, size=n)
    y = X @ theta_true + eps
    theta_hat = XtX_inv @ X.T @ y
    resid = y - X @ theta_hat
    s2 = resid @ resid / (n - p)
    se = np.sqrt(s2 * diag_V)
    lo = theta_hat - t_crit * se
    hi = theta_hat + t_crit * se
    covered += ((lo <= theta_true) & (theta_true <= hi)).astype(int)

print(f"target coverage  = {1 - alpha:.2f}")
print(f"empirical coverage per coefficient:")
for j in range(p):
    print(f"  theta_{j}:  {covered[j] / M:.4f}")

target coverage  = 0.95
empirical coverage per coefficient:
  theta_0:  0.9528
  theta_1:  0.9490
  theta_2:  0.9494


**Reading.** With M = 5000 the empirical coverage of each 95% CI should fall within Monte-Carlo error (≈ 0.6 pp) of 0.95. If we **violated** the assumptions (e.g. fat-tailed noise, heteroscedasticity), this number would drift away from 0.95 — a quick diagnostic in §10.

## 8. Hypothesis tests

### 8.1 The single-coefficient t-test

The most common test in regression: is coefficient j non-zero, i.e. does feature j carry any signal?

> H₀ : θⱼ = 0    vs.    H₁ : θⱼ ≠ 0.

Test statistic:

> t_j  :=  θ̂ⱼ  /  SE(θ̂ⱼ).   (8.1)

Under H₀, t_j ~ t(n − p) by Theorem 7.2. Reject at level α if |t_j| > t_{1 − α/2}(n − p).

The **p-value** is

> p_j  =  2 · ℙ_{T ~ t(n−p)}( T > |t_j| ).

### 8.2 The F-test for a sub-model

Sometimes we want to test that *several* coefficients are simultaneously zero, e.g. H₀ : θ₂ = θ₃ = θ₄ = 0. Let RSS_full be the residual sum of squares of the full model and RSS_red that of the *reduced* model with q < p free parameters. Define

> F  :=  [ (RSS_red − RSS_full) / (p − q) ]  /  [ RSS_full / (n − p) ].   (8.2)

Under H₀ + A1–A5, F ~ F(p − q, n − p). Reject for large F.

**Special case.** When q = 1 (only the intercept remains), this is the **overall F-test** of "any feature carries signal" — what most software reports as F-statistic at the bottom of a regression table.

*(Both tests are derived in any standard text, e.g. Hastie–Tibshirani–Friedman* Elements of Statistical Learning *§3.2; we omit the derivation to keep the focus on what each test tests.)*

## 9. The R² decomposition

Pythagoras (Corollary 5.4 of `02_mathematics.ipynb`) gave

> ‖y‖²  =  ‖ŷ\*‖²  +  ‖r\*‖².

When the design includes an intercept column 𝟙, an analogous decomposition holds for the **centred** target ỹ := y − ȳ · 𝟙:

> ‖ỹ‖²   =   ‖ŷ − ȳ · 𝟙‖²   +   ‖r̂‖².   (9.1)

Naming each piece:

| Symbol | Name | Formula |
|---|---|---|
| TSS | Total sum of squares | ∑ᵢ (yᵢ − ȳ)² |
| ESS | Explained sum of squares | ∑ᵢ (ŷᵢ − ȳ)² |
| RSS | Residual sum of squares | ∑ᵢ (yᵢ − ŷᵢ)² |

(9.1) says **TSS = ESS + RSS**.

### 9.1 Definition (coefficient of determination)

> R²  :=  1  −  RSS / TSS   =   ESS / TSS.   (9.2)

**Range.** From TSS = ESS + RSS and ESS, RSS ≥ 0: 0 ≤ R² ≤ 1, with R² = 1 iff RSS = 0 (perfect fit on the training set) and R² = 0 iff RSS = TSS (the model does no better than predicting the mean).

**Why bother?** R² is *scale-free*. MSE depends on the unit of y; R² says what fraction of the variance of y the model accounts for, regardless of the unit.

### 9.2 Caveats

1. **R² mechanically increases with more features.** Even random columns added to X lower the RSS (the minimum of a quadratic over a *larger* affine space cannot increase). The **adjusted R²** corrects for this:

   > R²_adj  =  1  −  (RSS / (n − p))  /  (TSS / (n − 1)).

2. **R² is a training-set quantity.** It says nothing about generalisation. A model with R² = 0.99 on training and R² = 0.10 on a held-out set has overfit — see `00_foundations/04_model_evaluation/`.

In [5]:
# Single fit on one realisation of y to compute R^2 and TSS = ESS + RSS numerically.
eps = rng.normal(0, sigma, size=n)
y = X @ theta_true + eps
theta_hat = np.linalg.solve(X.T @ X, X.T @ y)
y_hat = X @ theta_hat
y_bar = y.mean()

TSS = float(np.sum((y - y_bar) ** 2))
ESS = float(np.sum((y_hat - y_bar) ** 2))
RSS = float(np.sum((y - y_hat) ** 2))
R2  = 1 - RSS / TSS

print(f"TSS         = {TSS:.4f}")
print(f"ESS         = {ESS:.4f}")
print(f"RSS         = {RSS:.4f}")
print(f"ESS + RSS   = {ESS + RSS:.4f}   (should equal TSS)")
print(f"R^2         = {R2:.4f}")
print(f"identity holds within {abs(ESS + RSS - TSS):.2e}")

TSS         = 744.2837
ESS         = 701.8720
RSS         = 42.4117
ESS + RSS   = 744.2837   (should equal TSS)
R^2         = 0.9430
identity holds within 5.68e-13


## 10. When the assumptions break

Every theorem above carries the tag *under A1–A5*. The table catalogues what goes wrong when each assumption is violated, and points to the diagnostic that detects it.

| Violation | Symptom | Damage | Diagnostic |
|---|---|---|---|
| **A1 fails** (rank(X) < p, multicollinearity) | Two or more columns of X near-linearly dependent | (XᵀX)⁻¹ blows up → huge SEs, unstable signs | Variance inflation factors (VIF); condition number of X; correlation matrix |
| **A2 fails** (𝔼[ε \| X] ≠ 0, e.g. omitted variable, simultaneity) | Bias in θ̂ | θ̂ inconsistent — no amount of data fixes it | Residual plots vs. omitted predictors; instrumental-variable methods |
| **A3 fails** (heteroscedasticity, Var(εᵢ) depends on xᵢ) | Errors fan out in residual-vs-fitted plot | OLS still unbiased, but Var(θ̂) ≠ σ² (XᵀX)⁻¹ → CIs and p-values wrong | Breusch–Pagan / White test; heteroscedasticity-robust (HC) standard errors |
| **A4 fails** (autocorrelation, e.g. time-series) | Adjacent residuals correlated | Same as A3: SEs wrong | Durbin–Watson test; Newey–West standard errors |
| **A5 fails** (non-normal noise) | Heavy-tailed or skewed residuals | θ̂ still has correct mean & variance (Theorems 3.1, 4.1 only need A1–A4); but the t / F **exact** distributions in §7–§8 break | QQ-plot of residuals; rely on the **asymptotic** Gaussian via the CLT when n is large enough |

**The mental model.** A1 + A2 deliver **unbiasedness** (Theorem 3.1). A3 + A4 deliver the **variance formula** (Theorem 4.1) and Gauss–Markov optimality (Theorem 5.2). A5 delivers the **exact small-sample distribution** (Theorem 6.3) on which t-tests, F-tests, and CIs depend.

When A5 fails but A1–A4 hold and n is large, the **central limit theorem** rescues CIs and tests *asymptotically* — replace t-quantiles with normal quantiles. When A3 or A4 fail, the point estimates are still good but the *uncertainty quantification* is corrupted; the textbook fix is robust standard errors.

Algorithms in later notebooks of module 01 attack the A1 failure: **Ridge regression** (`03_ridge_regression/`) replaces (XᵀX)⁻¹ with (XᵀX + λIₚ)⁻¹, trading a little bias for a large variance reduction; **Lasso** (`04_lasso_regression/`) does the same and additionally zeroes out coefficients.

## Takeaway

- **Setup.**   X fixed, y = X θ + ε with ε random. θ̂ random because θ̂ = θ + (XᵀX)⁻¹ Xᵀ ε (sampling identity (3.1)).
- **Unbiasedness (Theorem 3.1).**   Under A1 + A2: 𝔼[θ̂] = θ.
- **Variance (Theorem 4.1).**   Under A1 + A2 + A3 + A4: Var(θ̂) = σ² · (XᵀX)⁻¹.
- **Gauss–Markov (Theorem 5.2).**   OLS is the unique BLUE: minimum variance among all linear unbiased estimators.
- **σ² estimator (Theorem 6.2).**   s² = RSS / (n − p) is unbiased; the n − p divisor counts residual degrees of freedom.
- **Sampling distribution (Theorem 6.3).**   Adding A5: θ̂ ~ 𝒩(θ, σ² (XᵀX)⁻¹), RSS/σ² ~ χ²(n − p), and the two are independent.
- **Inference.**   t_j = θ̂ⱼ / SE(θ̂ⱼ) ~ t(n − p) → confidence intervals (7.2) and the coefficient t-test (8.1); F-statistic (8.2) for sub-model tests.
- **R² (eqs. 9.1–9.2).**   TSS = ESS + RSS (centred Pythagoras); R² = 1 − RSS / TSS is the variance fraction explained on the training set. Adjusted R² penalises model size.
- **Failure modes (§10).**   A1 fails → variance inflation; A2 fails → bias; A3 / A4 fail → wrong standard errors; A5 fails → exact tests invalid, asymptotic tests still work.

Next: `05_hands_on_programming.ipynb` — implement everything we have derived (closed-form OLS, gradient descent OLS, residual variance, standard errors, R²) from scratch in NumPy, then cross-check against `sklearn.linear_model.LinearRegression` on a real dataset.